# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Loaded dataset: {metadata.name}\n")
print("Description:")
print(metadata.description)

# Print available dataset-level author(s) and citation(s), referenced by @id
print("\nAuthors (by @id):")
if hasattr(metadata, 'author'):
    if isinstance(metadata.author, list):
        for author in metadata.author:
            print(f"  - {author['@id'] if isinstance(author, dict) and '@id' in author else author}")
    else:
        print(f"  - {metadata.author}")

print("\nCitations (by @id):")
if hasattr(metadata, 'citation'):
    if isinstance(metadata.citation, list):
        for citation in metadata.citation:
            print(f"  - {citation['@id'] if isinstance(citation, dict) and '@id' in citation else citation}")
    else:
        print(f"  - {metadata.citation}")

## 2. Data Overview
Review available record sets, fields, and their IDs.
All entities are referenced by their `@id`. 

In [ ]:
# Fetch all record sets (by @id)
record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name', '[no name]')}")
    print(f"  Description: {rs.get('description', '[no description]')}")
    # List fields
    print("  Fields (@id):")
    for field in rs.get('field', []):
        if isinstance(field, dict) and '@id' in field:
            print(f"    - {field['@id']}")
        else:
            print(f"    - {field}")
    print("")
# Pick the first record set's @id for further exploration
if record_sets:
    primary_record_set_id = record_sets[0]['@id']
else:
    raise ValueError("No record sets found.")

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis. Only use their `@id` for references, as shown above.

In [ ]:
# Prepare DataFrames for all record sets by @id
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))  # list of dicts keyed by field @id
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[rs_id].columns.tolist()}")
        display(dataframes[rs_id].head(2))
    else:
        print("  No records loaded for this RecordSet.")

# Preview one of the dataframes (the primary record set)
df_main = dataframes[primary_record_set_id]
print("\nSample rows from main record set:")
display(df_main.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping.

We will:
- Filter records by a numeric field
- Normalize this field
- Group by a categorical field

**All columns are referenced by their `@id`.**

In [ ]:
# Inspect column names (fields' @id) in the main DataFrame
print("Available columns in the main record set:")
for idx, field in enumerate(df_main.columns):
    print(f"  {idx}. {field}")

# Let's choose a numeric field @id (for example, age at diagnosis)
# If unsure, display some records to pick a likely numeric field.
print("\nSample data:")
display(df_main.head())

# Assume '@id' for Age at 2nd primary CRC diagnosis is something like 'https://api.app.sen.science/frontiers/7862866/age_second_crc'
# You should replace these IDs with the actual ones found in your column list above.
# For example purposes, let's select the first numeric-looking column.
# Pick first float/integer or name-containing 'age'.
import re
numeric_field_id = None
for col in df_main.columns:
    # Heuristic: try to find a likely 'age' or integer column
    col_values = df_main[col].dropna().values
    if len(col_values) > 0 and isinstance(col_values[0], (int, float)):
        numeric_field_id = col
        break
    if numeric_field_id is None and re.search('age', col, re.IGNORECASE):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print("No numeric field found for EDA. Please check column names and pick an appropriate one.")

print(f"\nUsing numeric field for filtering and normalization: {numeric_field_id}")

# Filtering: records with value > threshold (e.g. threshold=50 for age)
threshold = 50
filtered_df = df_main[df_main[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalizing the numeric field (z-score)
normalized_col = numeric_field_id + "_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized column '{normalized_col}':")
display(filtered_df[[numeric_field_id, normalized_col]].head())

# Group by a categorical field. Pick a field likely representing sex or cancer type by @id
group_field_id = None
for col in df_main.columns:
    if re.search('sex|gender|site|location|cancer', col, re.IGNORECASE):
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    display(grouped_df)
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize distributions or relationships from the processed dataset using matplotlib or seaborn.

In [ ]:
# Simple histogram and boxplot of the numeric field in the filtered dataset
import matplotlib.pyplot as plt

if numeric_field_id and len(filtered_df) > 0:
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    filtered_df[numeric_field_id].hist(bins=15)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")

    plt.subplot(1, 2, 2)
    filtered_df.boxplot(column=numeric_field_id)
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.show()
    
    # If grouping field exists, plot mean numeric value by group
    if group_field_id:
        plt.figure(figsize=(6,4))
        grouped_df.plot.bar(x=group_field_id, y=numeric_field_id, legend=False, ax=plt.gca())
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load Croissant metadata and records via the `mlcroissant` library
- Explore available record sets and their fields by `@id`
- Extract records as DataFrames and reference all schema elements by their `@id`
- Apply EDA such as filtering, normalization, and grouping using selected schema fields
- Visualize the distribution and groupwise statistics of a numeric field

**Tip:** All entities in Croissant datasets should be referenced by their `@id` for reproducible analysis. Explore additional record sets and fields as needed using the code templates provided.
